# GridSight SMARD source profile

This notebook reproduces the Phase 2 structural and value-profile checks for all six immutable SMARD snapshots. The tested package code does the work; the notebook presents its results without modifying raw data.

In [ ]:
from pathlib import Path

import pandas as pd

from gridsight.ingestion.source_profiler import (
    profile_registered_snapshots,
    validate_source_profiles,
)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'configs' / 'smard_exports.json').is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG = PROJECT_ROOT / 'configs' / 'smard_exports.json'
MANIFEST = PROJECT_ROOT / 'data' / 'manifests' / 'smard_source_manifest.csv'
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'

In [ ]:
report = profile_registered_snapshots(CONFIG, MANIFEST, RAW_DIR)
violations = validate_source_profiles(report)
assert not violations, '\n'.join(violations)
print(f'Phase 2 source-profile checks passed for {len(report.snapshots)} snapshots.')

In [ ]:
summary_rows = []
for snapshot in report.snapshots:
    target = snapshot.target_profile
    summary_rows.append(
        {
            'export_id': snapshot.export_id,
            'rows': snapshot.row_count,
            'columns': snapshot.column_count,
            'unique_local_starts': snapshot.unique_start_count,
            'repeated_local_groups': snapshot.repeated_start_groups,
            'sha_matches_manifest': snapshot.sha_matches_manifest,
            'target_minimum': None if target is None else target.minimum,
            'target_maximum': None if target is None else target.maximum,
            'target_negative_rows': None if target is None else target.negative_count,
            'target_marker_rows': None if target is None else target.marker_count,
        }
    )

pd.DataFrame(summary_rows)

In [ ]:
pd.DataFrame(
    [
        {'source_category': category, 'schemas_match': compatible}
        for category, compatible in report.schema_compatible.items()
    ]
)

In [ ]:
marker_rows = []
for snapshot in report.snapshots:
    for measure in snapshot.measures:
        if measure.marker_count:
            marker_rows.append(
                {
                    'export_id': snapshot.export_id,
                    'measure': measure.name,
                    'marker_counts': measure.marker_counts,
                }
            )

pd.DataFrame(marker_rows)